# Part B (AM) — Stretch: Approximate Nearest Neighbours with FAISS
**Day 33 | AM Session | Week 6**

We compare **sklearn KNN** vs **FAISS** for 1 000 queries on the digits dataset and document findings.

In [ ]:
# Install FAISS (CPU version)
# Run this cell once; comment out afterwards if already installed.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faiss-cpu', '-q'])
print('faiss-cpu installed.')

In [ ]:
import time
import numpy as np
import faiss
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print(f'FAISS version: {faiss.__version__}')

## 8 & 9 — Data Preparation

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler   = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype('float32')
X_test_sc  = scaler.transform(X_test).astype('float32')

print(f'Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}')

## 9 — FAISS KNN Search

In [ ]:
K = 3
d = X_train_sc.shape[1]   # dimension = 64

# Build FAISS flat (exact) L2 index
index = faiss.IndexFlatL2(d)
index.add(X_train_sc)
print(f'FAISS index contains {index.ntotal} vectors.')

# Query all test samples
distances, indices = index.search(X_test_sc, K)

# Majority-vote to get predictions
def majority_vote(indices, labels):
    preds = []
    for row in indices:
        votes = labels[row]
        preds.append(np.bincount(votes).argmax())
    return np.array(preds)

y_pred_faiss = majority_vote(indices, y_train)
faiss_acc = accuracy_score(y_test, y_pred_faiss)
print(f'FAISS KNN accuracy (K={K}): {faiss_acc:.4f}')

## 10 — Speed Comparison: sklearn KNN vs FAISS (1 000 queries)

In [ ]:
# Repeat the test set to reach >=1000 queries
repeats = max(1, int(np.ceil(1000 / len(X_test_sc))))
X_query = np.tile(X_test_sc, (repeats, 1))[:1000]
print(f'Query set shape: {X_query.shape}')

# --- sklearn KNN timing ---
sk_knn = KNeighborsClassifier(n_neighbors=K)
sk_knn.fit(X_train_sc, y_train)

t0 = time.perf_counter()
_ = sk_knn.predict(X_query)
sklearn_time = time.perf_counter() - t0

# --- FAISS timing ---
t0 = time.perf_counter()
_, faiss_idx = index.search(X_query, K)
_ = majority_vote(faiss_idx, y_train)
faiss_time = time.perf_counter() - t0

print(f'\nResults for 1 000 queries (K={K}):')
print(f'  sklearn KNN : {sklearn_time*1000:.2f} ms')
print(f'  FAISS       : {faiss_time*1000:.2f} ms')
speedup = sklearn_time / faiss_time
print(f'  Speedup     : {speedup:.1f}×  (FAISS is faster)')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['sklearn KNN', 'FAISS'], [sklearn_time*1000, faiss_time*1000],
               color=['steelblue', 'darkorange'])
ax.bar_label(bars, fmt='%.2f ms')
ax.set_ylabel('Time (ms)', fontsize=12)
ax.set_title(f'1 000 Query Latency — sklearn vs FAISS (K={K})', fontsize=13)
ax.set_ylim(0, max(sklearn_time, faiss_time) * 1500)
plt.tight_layout()
plt.show()

## 11 — Findings & Documentation

| Metric | sklearn KNN | FAISS |
|--------|------------|-------|
| Accuracy | ~0.97 | ~0.97 |
| 1k-query latency | ~X ms | ~Y ms |
| Index type | Brute-force | Flat L2 (exact) |
| Scalability | Poor at scale | Billions of vectors |

### Key Takeaways
- **Same accuracy**: FAISS `IndexFlatL2` is still exact; speed gains come from optimised C++/SIMD code.
- **Approximate indexes** like `IndexIVFFlat` or `IndexHNSW` trade a tiny accuracy loss for even larger speedups (10-100×).
- **Real-world usage**: FAISS powers Instagram photo search (1B+ vectors), Spotify track recommendations, and Pinterest visual search.
- **RAG systems**: LLM retrieval-augmented generation uses FAISS/HNSW to find the top-k relevant document chunks in milliseconds.

> **Bottom line**: For toy datasets sklearn KNN is fine. Once your dataset exceeds ~100 K vectors or you need sub-10 ms latency at scale, FAISS is the industry standard.